# Teil 4: Evaluation

## Modell neu trainieren

In [ ]:
!pip install kagglehub pandas scikit-learn matplotlib --quiet

import pandas as pd
import kagglehub
import os
import glob
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, ConfusionMatrixDisplay,
    precision_score, recall_score
)

path = kagglehub.dataset_download("andrewmvd/steam-reviews")
csv_dateien = glob.glob(os.path.join(path, "*.csv"))
df = pd.read_csv(csv_dateien[0], nrows=100000)

df_clean = df[['review_text', 'review_score']].dropna()
df_clean = df_clean[df_clean['review_text'].str.strip() != '']

X = df_clean['review_text']
y = df_clean['review_score']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

vectorizer = TfidfVectorizer(max_features=10000, stop_words='english')
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf  = vectorizer.transform(X_test)

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_test_tfidf)

print("Modell bereit.")

## Aufgabe 4.1: Aussagekräftige Felder

Das wichtigste Feld für das Modell ist `review_text`. Dieser Text enthält die eigentliche Meinung des Nutzers und ist die einzige Eingabe, auf der das Modell basiert. Felder wie `review_votes` habe ich weggelassen, da sie für die Bewertung nicht wirklich relevant sind. Um zu zeigen, welche Wörter im Text am meisten Einfluss haben, werden unten die stärksten positiven und negativen Merkmale ausgegeben.

In [ ]:
import numpy as np

feature_names = vectorizer.get_feature_names_out()
koeffizienten = model.coef_[0]

top_positiv = np.argsort(koeffizienten)[-10:][::-1]
top_negativ = np.argsort(koeffizienten)[:10]

print("Top 10 Wörter für positive Bewertungen:")
for i in top_positiv:
    print(f"  {feature_names[i]}: {koeffizienten[i]:.3f}")

print()
print("Top 10 Wörter für negative Bewertungen:")
for i in top_negativ:
    print(f"  {feature_names[i]}: {koeffizienten[i]:.3f}")

## Aufgabe 4.2: Messmetrik

Als Messmetrik brauche ich die Accuracy. Sie gibt an, wie viele Vorhersagen das Modell insgesamt korrekt getroffen hat. Da der Datensatz relativ ausgeglichen zwischen positiven und negativen Bewertungen ist, ist Accuracy hier eine sinnvolle Metrik.

In [ ]:
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc:.4f} ({acc*100:.2f}%)")

## Aufgabe 4.3: Wahrheitsmatrix, Recall und Precision

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=[-1, 1])

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Negativ (-1)', 'Positiv (1)'])
disp.plot(cmap='Blues')
plt.title('Wahrheitsmatrix')
plt.show()

precision = precision_score(y_test, y_pred, pos_label=1)
recall    = recall_score(y_test, y_pred, pos_label=1)

print(f"Precision (positive Klasse): {precision:.4f}")
print(f"Recall    (positive Klasse): {recall:.4f}")

## Aufgabe 4.4: Zusammenfassung

Das Modell hat eine hohe Accuracy und zeigt damit, dass `review_text` schon genug ist, um den `review_score` vorherzusagen. Wörter wie "great", "love" oder "best" führen klar zu positiven Vorhersagen, während "broken", "waste" oder "bad" auf negative Bewertungen hinweisen. Der hohe Recall zeigt, dass das Modell positive Bewertungen sehr gut erkennt. Eine Schwäche ist, dass ironsiche oder sarkastische Texte z.B. "10/10 would uninstall again" das Modell verwirren können, weil es nur auf einzelnen Wörtern basiert und keinen ganzen Text Kontext versteht.